# Notebook 02 — Embedding Fine-Tuning with Contrastive Learning

We fine-tune a sentence-transformer model using a triplet loss so that semantically similar sentences have closer embeddings.

In [ ]:
# !pip install sentence-transformers datasets

## 1. Load a sentence pair dataset

In [ ]:
from datasets import load_dataset

# NLI dataset provides (anchor, positive, negative) triplets
dataset = load_dataset("sentence-transformers/all-nli", "triplet", split="train[:5000]")
print(dataset[0])

## 2. Prepare training examples

In [ ]:
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader

model = SentenceTransformer("all-MiniLM-L6-v2")

train_examples = [
    InputExample(texts=[row["anchor"], row["positive"], row["negative"]])
    for row in dataset
]
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=32)
train_loss = losses.TripletLoss(model=model)
print(f"Training on {len(train_examples)} triplets")

## 3. Fine-tune

In [ ]:
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=1,
    warmup_steps=100,
    output_path="./fine_tuned_embedder",
    show_progress_bar=True,
)
print("Fine-tuning complete. Model saved to ./fine_tuned_embedder")

## 4. Evaluate semantic similarity

In [ ]:
from sentence_transformers import util

sentences_a = ["The cat sat on the mat.", "A dog played in the park."]
sentences_b = ["A feline rested on a rug.", "Children ran in the garden."]

embeddings_a = model.encode(sentences_a)
embeddings_b = model.encode(sentences_b)

scores = util.cos_sim(embeddings_a, embeddings_b)
print("Cosine similarity matrix:")
print(scores)